# Flexible Classification Neural Network Class

In [1]:
# import packages
import numpy as np
import pandas as pd
print("import success")

import success


## Misc Functions

In [37]:
class ClassificationNeuralNetwork:
    def __init__(self, layer_list):
        self.layers = []
        self.activation = None
        for i in range(len(layer_list) - 1):
            scale = np.sqrt(2.0 / layer_list[i]) * 0.5 
            self.layers.append(np.random.randn(layer_list[i + 1], layer_list[i] + 1) * scale)

    @staticmethod
    def cross_entropy_derivative(logits, target):
        max_logits = np.max(logits, axis=0, keepdims=True)
        exps = np.exp(logits - max_logits)
        y_hat = exps / np.sum(exps, axis=0, keepdims=True)
        return y_hat - target 

    @staticmethod
    def cross_entropy_error(output, target):
        max_output = np.max(output, axis=0, keepdims=True)
        exps = np.exp(output - max_output)
        probs = exps / np.sum(exps, axis=0, keepdims=True)
        
        probs = np.clip(probs, 1e-15, 1.0)
        
        loss_per_image = -np.sum(target * np.log(probs), axis=0)
        return np.mean(loss_per_image)

    @staticmethod
    def softmax(logits):
        exps = np.exp(logits - np.max(logits))
        return exps / np.sum(exps)
    
    def see_layers(self):
        for layer in self.layers:
            print(layer)

    def forward_pass(self, data_batch, target_batch, activation, error, error_derivative):
        self.activation = activation
        self.error = error
        self.error_derivative = error_derivative
        batch_size = data_batch.shape[1]
        output = {}
        ones = np.ones((1, batch_size))
        
        output["data"] = data_batch
        output["layer 1 sum"] = self.layers[0] @ np.vstack([data_batch, ones])
        output["layer 1 output"] = activation(output["layer 1 sum"])
        
        for i in range(1, len(self.layers) - 1):
            output[f"layer {i+1} sum"] = self.layers[i] @ np.vstack([output[f"layer {i} output"], ones])
            output[f"layer {i+1} output"] = activation(output[f"layer {i+1} sum"])
            
        output[f"layer {len(self.layers)} output"] = self.layers[-1] @ np.vstack([output[f"layer {len(self.layers) - 1} output"], ones])
        output["error gradient"] = error_derivative(output[f"layer {len(self.layers)} output"], target_batch)
        output["error"] = error(output[f"layer {len(self.layers)} output"], target_batch).item()
        
        return output
        
    def back_prop(self, fpd, learning_rate, activation_derivative):
        batch_size = fpd["data"].shape[1]
        ones = np.ones((1, batch_size))

        delE_delSlast = fpd["error gradient"]
        delE_delWlast = (delE_delSlast @ np.vstack([fpd[f"layer {len(self.layers) - 1} output"], ones]).T) / batch_size
        delE_delWlast = np.clip(delE_delWlast, -1.0, 1.0)
        self.layers[-1] = self.layers[-1] - (learning_rate * delE_delWlast)

        for i in range(len(self.layers) - 1, 1, -1):
            delE_delOi = self.layers[i][:, :-1].T @ delE_delSlast
            delE_delSlast = delE_delOi * activation_derivative(fpd[f"layer {i} output"])
            delE_delSlast = np.clip(delE_delSlast, -1.0, 1.0)
            delE_delWi = (delE_delSlast @ np.vstack([fpd[f"layer {i-1} output"], ones]).T) / batch_size
            delE_delWi = np.clip(delE_delWi, -1.0, 1.0)
            self.layers[i-1] = self.layers[i-1] - (learning_rate * delE_delWi)

        delE_delO1 = self.layers[1][:, :-1].T @ delE_delSlast
        delE_delS1 = delE_delO1 * activation_derivative(fpd[f"layer 1 output"])
        delE_delW1 = (delE_delS1 @ np.vstack([fpd["data"], ones]).T) / batch_size
        delE_delW1 = np.clip(delE_delW1, -1.0, 1.0)
        self.layers[0] = self.layers[0] - (learning_rate * delE_delW1)

    def train(self, epochs, dataset, targets, activation, learning_rate, activation_derivative, batch_size=32):
        curr_lr = learning_rate
        for epoch in range(epochs):
            batch_count = 0
            indices = np.arange(len(dataset))
            np.random.shuffle(indices)
            dataset = dataset[indices]
            targets = targets[indices]
            total_error = 0
            if (epoch + 1) % 50 == 0:
                curr_lr *= 0.5
            for i in range(0, len(dataset), batch_size):
                batch_count += 1
                x_batch = dataset[i : i + batch_size].T
                y_labels = targets[i : i + batch_size]
                y_batch = np.eye(10)[y_labels].T
                d = self.forward_pass(
                    x_batch, y_batch, 
                    activation, 
                    ClassificationNeuralNetwork.cross_entropy_error, 
                    ClassificationNeuralNetwork.cross_entropy_derivative
                )
                self.back_prop(d, curr_lr, activation_derivative)
                total_error += d["error"]
            print(f"Avg Error for epoch {epoch + 1}: {total_error / batch_count}")

    def predict(self, dataset, targets):
        count = 0
        for i in range(len(dataset)):
            d = self.forward_pass(dataset[i].reshape(-1, 1), 
                                  np.eye(10)[targets[i]].reshape(-1, 1), 
                                  self.activation, 
                                  ClassificationNeuralNetwork.cross_entropy_error, 
                                  ClassificationNeuralNetwork.cross_entropy_derivative)
            output = ClassificationNeuralNetwork.softmax(d[f"layer {len(self.layers)} output"])
            prediction = np.argmax(output, axis=0)
            if prediction == targets[i]:
                count += 1
        return count / len(dataset)

In [3]:
df = pd.read_csv("train.csv")
df.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Initialise Activation Function and Derivative

In [23]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(y):
    return y * (1 - y)

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def leaky_relu(x):
    return np.where(x > 0, x, x * 0.01)

def leaky_relu_derivative(x):
    return np.where(x > 0, 1.0, 0.01)

## Prepare Data

In [6]:
data_df = df.drop(["label"], axis=1)
data_df.head()

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
df[:2000]["label"].value_counts()

label
2    237
1    217
6    211
4    203
9    198
0    196
7    194
5    187
8    184
3    173
Name: count, dtype: int64

In [24]:
data = data_df.to_numpy()
targets = df["label"].to_numpy()

print(data.shape)
print(targets.shape)

(42000, 784)
(42000,)


## Model Training

In [42]:
model = ClassificationNeuralNetwork([784, 1024, 1024, 512, 256, 128, 64, 32, 10])

In [ ]:
model.train(200, data[:2000], targets[:2000], leaky_relu, 0.001, leaky_relu_derivative)

Avg Error for epoch 1: 2.2026350527935206
Avg Error for epoch 2: 1.84353090066595
Avg Error for epoch 3: 1.4780707301877505
Avg Error for epoch 4: 1.0810614902940012
Avg Error for epoch 5: 0.7797627379422647
Avg Error for epoch 6: 0.5887220509502497
Avg Error for epoch 7: 0.5401842717451796
Avg Error for epoch 8: 0.43378528369090885


In [ ]:
df[3000:4001]["label"].value_counts()

In [40]:
print(model.predict(data[3000:4001], targets[3000:4001]))

0.9010989010989011


In [41]:
print(model.predict(data[4000:5001], targets[4000:5001]))

0.9150849150849151
